# Machine Learning Research Paper Replication
----
> why to replicate a ML paper? To gain skills. (Download a paper -> implement it-> repeat.)
>
> To implement the paper code -> To learn how to use it -> have feature of utilizing/using it.
----

[TRANSFORMERS FOR IMAGE RECOGNITION](https://arxiv.org/pdf/2010.11929)




### Anatomy of research paper:
|No.|Section  |  What is it? |
|--|--|---|
|1.|Abstract|an overview / summary of the paper's main findings/contributions.|
|2.|Introduction|what is the paper's main problem? and details of previous methods used to try and solve it.|
|3.|Method|what steps did the researchers take when conducting their research? for example, what model(s), data sources, training setups were used?|
|4.|Results|what are the outcomes of the paper? if a new type of model or training setup was used, how did the results of findings compared to prev work? (experiment tracking is handy here.)|
|5.|Conclusion|what are the limitations of suggested methods? what are some next steps for the research community?|
|6.|References|what resources/papers did the researchers reference from?|
|7.|Appendix|are there any extra resources / findings to look at that weren't included in any of the prev sections?|

--------------------------------------------------------------------------------------------
Where to find machine learning papers:

1. [arXiv (for research papers)](https://arxiv.org/)  
2. [Twitter (example: AK Twitter profile)](https://x.com/_akhaliq)  
3. [GitHub (for code)](https://github.com/lucidrains/vit-pytorch)  
4. [Papers with Code (papers + implementations)](https://paperswithcode.com/)  
5. [Awesome Machine Learning (GitHub curated list)](https://github.com/josephmisiti/awesome-machine-learning)  
6. [Hugging Face Spaces + Models](https://huggingface.co/)  
7. [LinkedIn & Medium (ML authors)](https://towardsdatascience.com/)  
8. [OpenReview](https://openreview.net/)  
9. [Semantic Scholar](https://www.semanticscholar.org/)
---------------------------------------------------------------
To learn about transformers:

1. [Attention is all you need](https://arxiv.org/abs/1706.03762)

2. [illustrated transformer](https://jalammar.github.io/illustrated-transformer/)

## 00. Get Setup
import code we previously written from helper functions and going modular -> important libraries -> setup agnostic code.

In [ ]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

In [ ]:
try:
  from torchinfo import summary
except:
  !pip install torchinfo
  from torchinfo import summary

# import helper_functions
try:
  from helper_functions import set_seeds, download_data, plot_loss_curves
except ImportError:
  !git clone https://github.com/mrdbourke/pytorch-deep-learning/
  !mv pytorch-deep-learning/helper_functions.py .
  !rm -rf pytorch-deep-learning
  from helper_functions import set_seeds, download_data, plot_loss_curves

# Imort going_modular
try:
  from going_modular import data_setup, engine
except ImportError:
  !git clone https://github.com/esraalmaeeni/pytorch-zero-to-mastery-MyPractice/
  !mv pytorch-zero-to-mastery-MyPractice/going_modular .
  !rm -rf pytorch-zero-to-mastery-MyPractice
  from going_modular import data_setup, engine

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1. Get Data

In [ ]:
image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/refs/heads/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")

In [ ]:
train_dir = image_path / "train"
test_dir = image_path / "test"

## 2. Create Datasets and DataLoaders

In [ ]:
from going_modular import data_setup
from torchvision import transforms

# Create image size
IMG_SIZE = 224
BATCH_SIZE = 32

# Create transform pipline
manual_transforms = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                                        transforms.ToTensor()])

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=manual_transforms,
                                                                               batch_size=BATCH_SIZE)
len(train_dataloader), len(test_dataloader)

## 2.3 Visualize a single image

In [ ]:
import random
import os
from pathlib import Path

#image_path_list = list(Path(train_dir).glob("*/*.jpg"))
#random_image_path = random.choice(image_path_list)

#dataset = train_dataloader.dataset
#rand_int = random.randint(0, len(dataset) - 1)
#image_sample = dataset[rand_int]

image_batch, image_label = next(iter(train_dataloader))
image, label = image_batch[0], image_label[0]

import matplotlib.pyplot as plt
image = image.permute(1, 2, 0)
plt.imshow(image)
plt.axis("off")
plt.title(f"{class_names[label]} , {image.shape}")

## 3. Replicating ViT: Overview

Looking at a whole machine learning research paper can be intimidating.
So, in order to make it more understandable, we can break it down into smaller pieces:
* **Inputs:** What goes into the model? (in our case image tensors)
* **Outputs**: what comes out of the model/layer/block? (in our case, we want the model to output image classification labels)
* **Layers**: takes an input and manipulate it with a function (ex, self-attention)
* **Blocks**: collection of layers.
* **Model:** collection of blocks.
----------------------------
**Treasure haunting** through the paper to pick crucial information to build the ViT:

AN IMAGE IS WORTH 16X16 WORDS:  TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE
* The 8 equations of MLP and multihead self-attention.
* Fig. 1 Model overview.
* Table 1.

[![image.png](https://i.postimg.cc/Jzd39Sx7/image.png)](https://postimg.cc/VJnbtG22)

Figure 1: Model overview. We split an image into fixed-size patches, linearly embed each of them, add position embeddings, and feed the resulting sequence of vectors to a standard Transformer encoder. In order to perform classification, we use the standard approach of adding an extra learnable “classification token” to the sequence. The illustration of the Transformer encoder was inspired by Vaswani et al. (2017).


---------------
The MLP contains two layers with a GELU non-linearity.
$$
\begin{aligned}
\mathbf{z}_0 & =\left[\mathbf{x}_{\text {class }} ; \mathbf{x}_p^1 \mathbf{E} ; \mathbf{x}_p^2 \mathbf{E} ; \cdots ; \mathbf{x}_p^N \mathbf{E}\right]+\mathbf{E}_{p o s}, \\
\mathbf{z}_{\ell}^{\prime} & =\operatorname{MSA}\left(\operatorname{LN}\left(\mathbf{z}_{\ell-1}\right)\right)+\mathbf{z}_{\ell-1}, \\
\mathbf{z}_{\ell} & =\operatorname{MLP}\left(\operatorname{LN}\left(\mathbf{z}_{\ell}^{\prime}\right)\right)+\mathbf{z}_{\ell}^{\prime}, \\
\mathbf{y} & =\operatorname{LN}\left(\mathbf{z}_L^0\right)
\end{aligned}
$$
$$
\begin{aligned}
& \mathbf{E} \in \mathbb{R}^{\left(P^2 \cdot C\right) \times D}, \mathbf{E}_{p o s} \in \mathbb{R}^{(N+1) \times D} \\
& \ell=1 \ldots L \\
& \ell=1 \ldots L
\end{aligned}
$$

A Multihead Self-attention

Standard qkv self-attention (SA, Vaswani et al. (2017)) is a popular building block for neural architectures. For each element in an input sequence $\mathrm{z} \in \mathbb{R}^{N \times D}$, we compute a weighted sum over all values $\mathbf{v}$ in the sequence. The attention weights $A_{i j}$ are based on the pairwise similarity between two elements of the sequence and their respective query $\mathbf{q}^i$ and key $\mathbf{k}^j$ representations.
$$
\begin{aligned}
{[\mathbf{q}, \mathbf{k}, \mathbf{v}] } & =\mathbf{z} \mathbf{U}_{q k v} \\
A & =\operatorname{softmax}\left(\mathbf{q} \mathbf{k}^{\top} / \sqrt{D_h}\right) \\
\mathrm{SA}(\mathbf{z}) & =A \mathbf{v}
\end{aligned}
$$

Multihead self-attention (MSA) is an extension of SA in which we run $k$ self-attention operations, called "heads", in parallel, and project their concatenated outputs. To keep compute and number of parameters constant when changing $k, D_h$ (Eq. 5) is typically set to $D / k$.
$$
\operatorname{MSA}(\mathbf{z})=\left[\mathrm{SA}_1(z) ; \mathrm{SA}_2(z) ; \cdots ; \mathrm{SA}_k(z)\right] \mathbf{U}_{m s a} \quad \mathbf{U}_{m s a} \in \mathbb{R}^{k \cdot D_h \times D}
$$

#### Section 3.1 of the paper: Discription of various equations used

**Equation 1:**

An overview of the model is depicted in Figure 1. The standard Transformer receives as input a 1D sequence of token embeddings. To handle 2D images, we reshape the image $\mathbf{x} \in \mathbb{R}^{H \times W \times C}$ into a sequence of flattened 2D patches $\mathbf{x}_p \in \mathbb{R}^{N \times\left(P^2 \cdot C\right)}$, where $(H, W)$ is the resolution of the original image, $C$ is the number of channels, $(P, P)$ is the resolution of each image patch, and $N=H W / P^2$ is the resulting number of patches, which also serves as the effective input sequence length for the Transformer. The Transformer uses constant latent vector size $D$ through all of its layers, so we flatten the patches and map to $D$ dimensions with a trainable linear projection (Eq. 1). We refer to the output of this projection as the patch embeddings.










**Position embeddings **are added to the patch embeddings to retain positional information. We use standard learnable 1D position embeddings, since we have not observed significant performance gains from using more advanced 2D-aware position embeddings (Appendix D.4). The resulting sequence of embedding vectors serves as input to the encoder.

In pseudocode:
```python
x_input = [class_token, image_patch_1, image_patch_2,...image_path_N] + [class_token_pos + image_patch_1_pos + image_patch_2_pos,...image_patch_N_pos]

```



---





**Equation 2&3:**

The Transformer encoder (Vaswani et al., 2017) consists of alternating layers of multiheaded selfattention (MSA, see Appendix A) and MLP blocks (Eq. 2, 3). Layernorm (LN) is applied before every block, and residual connections after every block (Wang et al., 2019; Baevski \& Auli, 2019).

```python
# Equation 2
x_output_MSA_block = MSA_layer(LN_layer(x_input)) + x_input

#Equation 3
x_output_MLP_block = MLP_layer(LN_layer(x_output_MSA_block)) + x_output_MSA_block
```
> **residual connection** — the original input is added back to the output of the operation.
```Output = SomeFunction(x_input) + x_input```

> **Residual connections**  help gradients flow more easily during backpropagation and make it easier to learn identity functions when needed.


---




**Equation 4:**

Similar to BERT's [class] token, we prepend a learnable embedding to the sequence of embedded patches $\left(\mathrm{z}_0^0=\mathrm{x}_{\text {class }}\right)$, whose state at the output of the Transformer encoder $\left(\mathrm{z}_L^0\right)$ serves as the image representation $y$ (Eq. 4). Both during pre-training and fine-tuning, a classification head is attached to $\mathbf{z}_L^0$. The classification head is implemented by a MLP with one hidden layer at pre-training time and by a single linear layer at fine-tuning time.

MLP = one hidden layer at training time/ single linear layer at fine-tuning time.

```python
y = Linear_Layer(LN_layer(x_output_MLP_block))
```


> **Note:** To upload images: [Imgur](https://imgur.com), [Postimages](https://postimages.org), [Cloudinary](https://cloudinary.com), [Google Drive](https://drive.google.com), [GitHub](https://github.com).  
> To copy text as markdown: [Mathpix](https://mathpix.com).


Embeddings are vector representations of data — they translate complex inputs like words or image patches into numerical forms that models (like Transformers) can process.

#### Table 1: Details of Vision Transformer model variants.

| Model     | Layers | Hidden size *D* | MLP size | Heads | Params |
|-----------|--------|------------------|----------|--------|--------|
| ViT-Base  | 12     | 768              | 3072     | 12     | 86M    |
| ViT-Large | 24     | 1024             | 4096     | 16     | 307M   |
| ViT-Huge  | 32     | 1280             | 5120     | 16     | 632M   |

-----
`ViT-Base`, `ViT-Large`, and `ViT-Huge` are all different sizes of the same model architecture.
* Layers (blocks): the number of transformer encoder layers. (deeper models are more expressive.)
* Hidden size (vector D must be dividisible by heads): the embedding size throughout the architecture.
* MLP size(hidden size * 4 to avoid bottlenecking): the number of hidden units in MLP.
* Heads: the number of MSA.

>D % heads == 0	Equal head sizes, easier to compute
>
>D / heads ≈ 64–80	Empirically optimal attention power


> Tip: start with the smallest version, make it works then upscale if neede. (in our case we can start with ViT_base)

## 4. Equation 1: Split data into patches and creating the class, position, and patch embedding

Layers = inputs -> functions -> output
* input shape: (224, 224, 3) -> single image of (H*W*C)
* output shape: ????

### 4.1 Calculating input and output shape by hand

**Equation 1:**

An overview of the model is depicted in Figure 1. The standard Transformer receives as input a 1D sequence of token embeddings. To handle 2D images, we reshape the image $\mathbf{x} \in \mathbb{R}^{H \times W \times C}$ into a sequence of flattened 2D patches $\mathbf{x}_p \in \mathbb{R}^{N \times\left(P^2 \cdot C\right)}$, where $(H, W)$ is the resolution of the original image, $C$ is the number of channels, $(P, P)$ is the resolution of each image patch, and $N=H W / P^2$ is the resulting number of patches, which also serves as the effective input sequence length for the Transformer. The Transformer uses constant latent vector size $D$ through all of its layers, so we flatten the patches and map to $D$ dimensions with a trainable linear projection (Eq. 1). We refer to the output of this projection as the patch embeddings.



**Position embeddings **are added to the patch embeddings to retain positional information. We use standard learnable 1D position embeddings, since we have not observed significant performance gains from using more advanced 2D-aware position embeddings (Appendix D.4). The resulting sequence of embedding vectors serves as input to the encoder.

In pseudocode:
```python
x_input = [class_token, image_patch_1, image_patch_2,...image_path_N] + [class_token_pos + image_patch_1_pos + image_patch_2_pos,...image_patch_N_pos]

```

* Input shape: $H\times{W}\times{C}$ (Height x Width x color channels)
* Output shape: ${N\times\left(P^{2} \cdot C\right)}$

* Number of patches N = (Height * width) / p^2
* D = constant latent vector size = embed dimentions
* H
* W
* C
* P


In [ ]:
# Create example values
height = 224
width = 224
color_channels = 3
patch_size = 16

# Calculate the number of patches
number_of_patches = int((height * width) / patch_size**2)
number_of_patches

In [ ]:
# Input shape
embedding_layer_input_shape = (height, width, color_channels)

#output shape
embedding_layer_output_shape = (number_of_patches, patch_size**2 * color_channels)


print(f"Input shape (single 2D image): {embedding_layer_input_shape}")
print(f"Output shape (single 1D sequence of patches): {embedding_layer_output_shape} -> (number_of_patches, embedding_dimension)")

In [ ]:
image.shape

> It is all about the embedding. If can present data in a good learnable representation, the model architecture is going to learn efficiently.

In [ ]:
# Get the top row of the permuted image
plt.figure(figsize=(patch_size, patch_size))
plt.imshow(image[:patch_size,:,:])

In [ ]:
# setup code to plot top row as patches
img_size = 224
patch_size = 16
num_patches = img_size/patch_size
assert img_size % patch_size == 0, "Image size must be divisible by patch size"
print(f"Number of patches per row: {num_patches}\nPatch size: {patch_size} pixels x {patch_size} pixels")

# create a series of subplots
fig, axs = plt.subplots(nrows=1,
                        ncols=img_size // patch_size,
                        sharex=True,
                        sharey=True,
                        figsize=(patch_size, patch_size))

# Iterate through number of patches in the top row
for i, patch in enumerate(range(0, img_size, patch_size)): #range(from,to,steps of) i: sequence from 0 to the end divided by steps, patch: the ineger where each parition start from
  print(patch)
  print(i)
  axs[i].imshow(image[:patch_size,patch:patch+patch_size,:])
  axs[i].set_xlabel(i+1)   # Set x-axis label to the patch index (i+1)
  axs[i].set_xticks([]) # to disable x-axis tick marks
  axs[i].set_yticks([]) # to disable y-axis tick marks

#print(image[:patch_size,0:patch_size,:]) # To display the values of one patch embedding

In [ ]:
# Each pixel represents one color of 3 channels
one_pixel = image[0:1, 0:1, :]    # Select a single pixel (still in 3D shape)
plt.imshow(one_pixel)             # Display the pixel as an image
plt.title(str(one_pixel))         # Show pixel value as title (optional)
plt.axis("off")                   # Hide axes

In [ ]:
# Now converting the whole image to batches of size 16 x 16
plt.figure(figsize=(9, 9))
plt.imshow(image)
plt.axis("off")

In [ ]:
# Ensure image is divisible by patch_size
print(f"Number of total patches: {(img_size // patch_size) *  (img_size // patch_size)}\nNumber of patches per row/column: {img_size//patch_size}\nPatch size: {patch_size} pixels x {patch_size} pixels.")
assert img_size % patch_size == 0, "Image size must be divisible by patch size"

fig, axs = plt.subplots(nrows=img_size // patch_size,
                           ncols=img_size // patch_size,
                           figsize=(9,9),
                           sharex=True,   # ensures all patch subplots use the same x-axis scale (patch position)
                           sharey=True)

for i, col_patch in enumerate(range(0,img_size,patch_size)):
  for j, row_patch in enumerate(range(0,img_size,patch_size)):
    axs[i,j].imshow(image[row_patch:row_patch+patch_size,col_patch:col_patch+patch_size,:])
    axs[i, j].set_xlabel(j+1)   # Set x-axis label to the patch index (i+1)
    axs[i, j].set_ylabel(i +1,
                         rotation="horizontal",
                         horizontalalignment="right",
                         verticalalignment="center")
    axs[i, j].set_xticks([]) # to disable x-axis tick marks
    axs[i, j].set_yticks([])
    axs[i, j].label_outer()

fig.suptitle(f"{class_names[label]} patchified! ")
plt.tight_layout()
plt.show()


## Creating image patches and turning them into patch embeddings
using `torch.nn.Conv2d()` we can create image patches in one step, and setting the kernel size and stride parameters into `patch_size`

In [ ]:
# Create conv2d layer to turn image into patches of learnable feature maps (embeddings)
from torch import nn

# Set the patch size
patch_size = 16

conv2d = nn.Conv2d(in_channels=3,
                   out_channels=768,
                   kernel_size=patch_size,
                   stride=patch_size,
                   padding=0)
conv2d

In [ ]:
# View single image
plt.imshow(image)
plt.title(class_names[label])
plt.axis("off")

In [ ]:
print(image.shape)
permuted_image = image.permute(2,1,0)
print(permuted_image.shape)

In [ ]:
# pass the image through the conv2d layer
image_passed_conv2d = conv2d(permuted_image.unsqueeze(0))
print(image_passed_conv2d.shape,image_passed_conv2d) # patches -> now flatten into 1d patches

After passing a single image through `conv2d` layer
```python
torch.Size([1, 768, 14, 14]) # [batch_size, embedding_dim, feature_map_height, feature_map_width]
```

In [ ]:
# plot random convolutional features maps (embeddings)
import random
random_indexes = random.sample(range(0,768), k=5)
print(f"showing random convolutional feature maps from indexes: {random_indexes}")

fig, axs = plt.subplots(nrows=1,
                       ncols=5,
                       figsize=(15, 3))

for i, idx in enumerate(random_indexes):
  #patch_sample = image_passed_conv2d[0, random_indexes[i]].detach().cpu() # This is wrong, it will view pixels not patches
  patch_sample = image_passed_conv2d[:, idx,:,:].detach().cpu() # requires_grad = True -> use .detach().cpu()
  axs[i].imshow(patch_sample.squeeze())
  axs[i].set_xticks([])
  axs[i].set_yticks([])

fig.suptitle(f"{class_names[label]}")
plt.tight_layout()
plt.show()

> The visualizozation is 14 x 14 pixels, because the shape [1, 786, 14, 14]
So, there are 786 feature map of the size 14 x 14.as_integer_ratioWe need to turn them into series of convolutional feature map flatten into a sequence of patch embedding, to satisfy the  input criteria of ViT.

In [ ]:
# Get a single feature map in tensor form
single_feature_map = image_passed_conv2d[:,0,:,:]
single_feature_map

# 6. Flattening the patch embedding with `torch.nn.Flatten()`

In [ ]:
flatten_image = nn.Flatten(start_dim=2, end_dim=3)
Flatten_image = flatten_image(image_passed_conv2d)
Flatten_image.shape

```python
x_input = [class_token, image_patch_1, image_patch_2,...image_path_N] + [class_token_pos + image_patch_1_pos + image_patch_2_pos,...image_patch_N_pos]
```
The required output must be [196, 768] (number of patches, embedding shape)

In [ ]:
# To put everything together
plt.imshow(image)
plt.title(class_names[label])
plt.axis("off")
print(f"Original image shape: {image.shape}")

# turn image into feature map
image_out_of_conv2d = conv2d(permuted_image.unsqueeze(0))
print(f"Image feature map (patches) shape: {image_passed_conv2d.shape}")

# flatten feature map
image_out_of_flatten = flatten_image(image_out_of_conv2d)
print(f"Flatten image feature map shape: {image_out_of_flatten.shape}")

In [ ]:
# Rearange output of flattened image
print(f"{image_out_of_flatten.permute(0,2,1).shape} -> (batch_size, number_of_patches, embedding_size)")

In [ ]:
image_out_of_flatten_permuted = image_out_of_flatten.permute(0,2,1)
#print(image_out_of_flatten_permuted.shape)
#print(f"A single flattened feature map: {image_out_of_flatten_permuted[:,:,0]}")

# plot the flattened feature map
plt.figure(figsize=(22,22))
plt.imshow(image_out_of_flatten_permuted[:,:,0].detach().numpy())
plt.title(f"Flattened feature map shape: {image_out_of_flatten_permuted.shape}")
plt.axis("off")

In [ ]:
single_flatten_feature_map = image_out_of_flatten_permuted[:,:,0]
plt.figure(figsize=(22,22))
plt.imshow(single_flatten_feature_map.detach().numpy())
plt.title(f"Flattened feature map shape: {single_flatten_feature_map.shape}")
plt.axis("off")

### 4.5 Turning the ViT patch embedding layer intp a PyTorch module

we want this module to do a few things:
1. Create a class called `patchEmbedding` that inherits from `nn.Module`.
2. Initialize with appropriate hyperparameters, such as channels, dimension, embedding dimention, patch size.
3. Create a layer to turn the an image into embedding patches using `nn.conv2d()`.
4. Create a layer to flatten the feature maps of the output of the layer in 3.
5. Define a `forward()` function that defines the forward computation (e.g. pass through layer 3 and 4).
6. Make sure the output shape of the layer reflects the required output shape of the patch embedding.

In [ ]:
# 1. Create a class called PatchEmbedding
class PatchEmbedding(nn.Module):
  # 2. Initialize with appropriate hyperparameters
  def __init__(self,
               input_channels: int = 3,
               embedding_dim: int = 768,
               patch_size: int = 16):
    super().__init__()

    self.patch_size = patch_size # To access them later in the class

    # 3. Create a layer to turn the an image into embedding patches
    self.patcher = nn.Conv2d(in_channels=input_channels,
                          out_channels=embedding_dim,
                          kernel_size=patch_size,
                          stride=patch_size,
                          padding=0
                          )

    # 4. Create a layer to flatten the feature maps
    self.flatten = nn.Flatten(start_dim=2, end_dim=3)

  # 5. Define a forward() function
  def forward(self,x):
      # Create assertion to check that inputs are the correct shape
      image_resolution = x.shape[-1]
      assert image_resolution % patch_size == 0, "Input image size must be divisible by patch size"

      # Perform the forward pass
      x_patched = self.patcher(x)
      x_flattened = self.flatten(x_patched)
      # Make sure the output shape of the layer reflects the required output shape of the patch embedding
      return x_flattened.permute(0,2,1)

In [ ]:
set_seeds()

# Create instance of patch embedding layer
patchify = PatchEmbedding(input_channels=3,
                          embedding_dim=786,
                          patch_size=16)

image =  image.permute(2,1,0)
# Pass a single image through patch embedding layer
print(f"Input image size: {image.unsqueeze(0).shape}")
patch_embedded_image = patchify(image.unsqueeze(0)) # add extra batch dimention or it will throw an error
print(f"Output patch embedding sequence shape: {patch_embedded_image.shape}")

In [ ]:
rand_image_tensor = torch.randn(1, 3, 224, 224)
rand_image_tensor_bad = torch.randn(1, 3, 225, 225)

#patchify(rand_image_tensor_bad) # AssertionError: Input image size must be divisible by patch size

### 4.6 Creating the class token embedding
Goal: prepend a learnable class token to the start of the patch embedding

In [ ]:
patch_embedded_image

In [ ]:
# Get the patch size and embedding dimension
batch_size = patch_embedded_image.shape[0]
embedding_domension = patch_embedded_image.shape[-1]
batch_size, embedding_domension

In [ ]:
# Create class token embeddding as a learnable parameter that shares the same size as the
class_token = nn.Parameter(torch.ones(batch_size, 1, embedding_domension),
                           requires_grad=True)
class_token.shape

In [ ]:
patch_embedded_image.shape

In [ ]:
# Add the class token embedding to the front of the patch embedding
patch_embedded_image_with_class_embedding = torch.cat((class_token, patch_embedded_image),
                                                      dim=1) # Number of patches embedding

print(patch_embedded_image_with_class_embedding)
print(f"Sequence of patch embeddings with class token prepended shape: {patch_embedded_image_with_class_embedding.shape}")

### 4.7 Adding the position embedding
Goal: create a series of 1D learnable position embedding and add it to the sequence of patch embedding.

In [ ]:
# View the sequence of patch embedding with the prepended class embedding
patch_embedded_image_with_class_embedding, patch_embedded_image_with_class_embedding.shape

> Always check input and output to check everything is setup correctly.

In [ ]:
# Calculate number of N (number of patches)
number_of_patches = int((height * width) / patch_size**2)

# Get the embedding dimension
embedding_domension = patch_embedded_image_with_class_embedding.shape[-1]

# Create the learnable 1D position embedding
position_embedding = nn.Parameter(torch.ones(1,
                                             number_of_patches+1,
                                             embedding_domension),
                                  requires_grad=True)

position_embedding, position_embedding.shape

In [ ]:
patch_embedded_image_with_class_embedding.shape

In [ ]:
# Add the position embedding to the fron of patch with class embedding
patch_and_pos_embedding = position_embedding + patch_embedded_image_with_class_embedding # The values of pos add to the patch and class embedding, values change not the dimensions

print(patch_and_pos_embedding)
print(f"Sequence of patch embeddings with class and position token prepended shape: {patch_and_pos_embedding.shape}")

### 4.8 Putting it all together: from image to embedding
we have created a module to patch and flatten image, now we will put it in one cell.

In [ ]:
image.shape

In [ ]:
# Set seeds
set_seeds()

# 1. Set the patch size
patch_size = 16

# 2. print shape of the original image tensor and get the image dimensions
print(f"\nImage tensor shape: {image.shape}")
height, weight = image.shape[1], image.shape[-1]

# 3. Get image tensor and add a batch dimension
x = image.unsqueeze(0)
print(f"\nInput image shape: {x.shape}")

# 4. Create patch embedding layer
patch_embedding_layer = PatchEmbedding(input_channels=3,
                                       embedding_dim=768,
                                       patch_size=patch_size)

# 5. patch the image
embedded_patches = patch_embedding_layer(x)
print(f"\nThe patches embedding shape: {embedded_patches.shape}")

# Get the batch size and embedding dimension
batch_size = embedded_patches.shape[0]
embedding_dimension = embedded_patches.shape[-1]

# 6. Create class token
class_token = nn.Parameter(torch.randn(batch_size,1,embedding_dimension),
                              requires_grad=True)
print(f"\nThe class token shape: {class_token.shape}")

# 7. Add the class token to the patch embedding
class_patches_embedding = torch. cat((class_token, embedded_patches),
                                    dim=1)
print(f"\nThe embedded patches with class embedding shape: {class_patches_embedding.shape}")
print(class_patches_embedding)

# Get the number of patches
number_of_patches = ((height * weight) // patch_size ** 2)


# 8. Create pos embedding
pos_embedding = nn.Parameter(torch.randn(1, number_of_patches + 1, embedding_dimension),
                                requires_grad=True) # Make sure it is learnable

print(f"\nThe pos embedding shape: {pos_embedding.shape}")

# 9 . Add the pos embedding to the patch with class embedding
embedded_patches_with_class_and_pos_embedding = pos_embedding + class_patches_embedding
print(f"\nThe patches with class and pos embedding shape: {embedded_patches_with_class_and_pos_embedding.shape}")
print(embedded_patches_with_class_and_pos_embedding)


### 4.9 Putting it all together in one class

In [ ]:
class PatchEmbeddingWithClassAndPos(nn.Module):
    def __init__(self,
                 input_channels: int = 3,
                 embedding_dim: int = 768,
                 patch_size: int = 16,
                 image_size: int = 224):  # you can pass H or assume square
        super().__init__()
        self.patch_size = patch_size
        self.embedding_dim = embedding_dim
        self.num_patches = (image_size // patch_size) ** 2

        self.patcher = nn.Conv2d(in_channels=input_channels,
                                 out_channels=embedding_dim,
                                 kernel_size=patch_size,
                                 stride=patch_size,
                                 padding=0)

        self.flatten = nn.Flatten(start_dim=2, end_dim=3)

        # Register class token and pos embedding as learnable parameters
        self.class_token = nn.Parameter(torch.randn(1, 1, embedding_dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, embedding_dim))

    def forward(self, x):
        # x shape: [B, C, H, W]
        assert x.shape[-1] % self.patch_size == 0, "Image size must be divisible by patch size"

        x = self.patcher(x)            # [B, D, H/P, W/P]
        x = self.flatten(x)            # [B, D, N]
        x = x.permute(0, 2, 1)         # [B, N, D]

        batch_size = x.shape[0]
        class_token = self.class_token.expand(batch_size, -1, -1)  # [B, 1, D]
        x = torch.cat((class_token, x), dim=1)                     # [B, N+1, D]

        print(x)

        x = x + self.pos_embedding                                  # [B, N+1, D]

        print(f"Final embedding shape: {x.shape}")
        print(x)
        return x

In [ ]:
patching = PatchEmbeddingWithClassAndPos(input_channels=3,
                         embedding_dim=768,
                         patch_size=16)
print(permuted_image.shape)
image_patches = patching(permuted_image.unsqueeze(0))

## 5. Equation 2: Multihead Self-Attention (MSA block)

* Multihead self-attention: which part of sequence should pay most attention to itself?
  * in our case, we have sequence pf patch embedding, which patch signifucantly relates to another patch.
  * We want the neural network to learn this relationship/representation.

  * Layer norm: a technique to normalize the distribution of intermediate layer. It enables smoother gradients, faster training, and better generalization accuracy.

    * normalize values over D dimentions (the embedding dimension) -> so everything has the same mean and standard divation. it is like making all the the stairs in a staircase the same size.



In [ ]:
class MultiHeadSelfAttentionBlock(nn.Module):
    """
    Create a multi-head self-attention block (MSA block).

    Applies LayerNorm → Multi-Head Self-Attention → Residual Connection.

    Args:
        embedding_dim (int): Dimensionality of input embeddings.
        num_heads (int): Number of attention heads.
        dropout
    """
    def __init__(self, embedding_dim: int = 768, num_heads: int = 12, attn_dropout: int = 0 ):
        super().__init__()
        self.layer_norm = nn.LayerNorm(embedding_dim)
        self.multhead_attn = nn.MultiheadAttention(embed_dim=embedding_dim,
                                         num_heads=num_heads,
                                         dropout = attn_dropout,
                                         batch_first=True)  # So the output tensor are provided as (batch, seq, feature) -> (batch, number_of_batches, embedding_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_norm = self.layer_norm(x)
        attn_output, _ = self.multhead_attn(query=x_norm,
                                  key=x_norm,
                                  value=x_norm,
                                  need_weights=False) # attn_output, attn_output_weights = multihead_attn(query, key, value)
        #return attn_output + x    # Residual connection
        return attn_output

In [ ]:
# Create an instance MSA block
MSA_block = MultiHeadSelfAttentionBlock(embedding_dim=768,
                                        num_heads=12,
                                        attn_dropout=0)

# Pass the patch and position image embedding sequence through MSA block
attn_embedded_patches = MSA_block(image_patches)
print(f"Input shape of MSA block: {attn_embedded_patches.shape}")
print(attn_embedded_patches)

## 6. Equation 3: multiple perceptron block (MLP block)

* **MLP** = The MLP contains two layers with a GELU non-linearity.
* MLP: a broad term for a block with series of layers, layers can be multiple or even one hidden layer.
* Layers can mean fully_connected, dense, linear, feed_forward..all mean same thing.
* in PyTorch, they are called `torch.nn.Linear()`, and in tensorFlow `tf.Keras.layers.Dense()`
* The GELU nonlinearity weights inputs by their value, rather than gates inputs by their sign as in ReLUs (x1x>0).


In pseudocode:

```python
# MLP
x = linear -> non-linear -> dropout -> linear -> dropout
````

In [ ]:
class MLPBlock(nn.Module):
  """ Create  the multiple perceptron block (MLP block).
  """
  def __init__(self, embedding_dim:int=768, mlp_size:int = 3072, dropout:int=0.1):
    super().__init__()

    # Create the norm layer (LN)
    self.layer_norm = nn.LayerNorm(normalized_shape=embedding_dim)

    # Create the MLP
    self.mlp_layer = nn.Sequential(
        nn.Linear(in_features= embedding_dim,
                  out_features=mlp_size),
        nn.GELU(),
        nn.Dropout(p=dropout),
        nn.Linear(in_features=mlp_size,
                  out_features=embedding_dim),
        nn.Dropout(p=dropout)
    )

  def forward(self, x:torch.Tensor) -> torch.Tensor:
    x_norm = self.layer_norm(x)
    x_mlp = self.mlp_layer(x_norm)
    return x_mlp

In [ ]:
# Create an instance of MLP block
mlp_block = MLPBlock(embedding_dim=768,
                     mlp_size=3072,
                     dropout=0.1)

# Pass the output of MSABlock through the MLPBlock
patched_image_through_mlp_block = mlp_block(attn_embedded_patches)
print(f"Input shape of MLP block: {patched_image_through_mlp_block.shape}")
print(patched_image_through_mlp_block)

In [ ]:
print(f"Input shape of MSA block: {attn_embedded_patches.shape}")
print(f"Input shape of MLP block: {patched_image_through_mlp_block.shape}")

## 7. Creating the transformer encoder
The transformer encoder is a combination of an alterating blocks of MSA and MLP.

Adding the residual connections after each block.

* Encoder = turn a sequence into learnable representation.
* Decoder = go from learnable representation back to some sort of sequence.
* Residual connections = add a layer(s) input to its subsequent output, this enables the creation of deeper networks (preventing weights from getting too small.)

In pseudocode:
```python
x_input -> MSA_block -> [MSA_block_output + x_input] -> MLP_block -> [MLP_block_output + MSA_block_output + x_input] -> ...
```


### 7.1 Create a custom Transformer Encoder block

In [ ]:
class TransformerEncoderBlock(nn.Module):
  def __init__(self,
               embedding_dim:int=768,
               num_heads:int=12,
               mlp_size:int=3072,
               mlp_dropout:int=0.1,
               attn_dropout:int=0):

    super().__init__()

    # Create MSA block
    self.msa_block = MultiHeadSelfAttentionBlock(embedding_dim=embedding_dim,
                                                 num_heads=num_heads,
                                                 attn_dropout=attn_dropout)

    self.mlp_block = MLPBlock(embedding_dim=embedding_dim,
                              mlp_size=mlp_size,
                              dropout=mlp_dropout)

  def forward(self, x):
    x = self.msa_block(x) + x   # Add skip connections
    x = self.mlp_block(x) + x   # Add skip connections
    return x

In [ ]:
# Create an instance of TransformerEncoderBlock()
transformer_encoder_block = TransformerEncoderBlock()

# Get a summary using torchinfo.summary
from torchinfo import summary
summary(model=transformer_encoder_block,
        input_size=(1, 197, 768),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

### 7.2 Create a transformer Encoder Layer with in-built PyTorch layers


In [ ]:
# Create the same as the class TransformerEncoderBlock() with torch.nn.TransformerEncoderLayer
torch_transformer_encoder_block = nn.TransformerEncoderLayer(d_model=768,
                                                       nhead=12,
                                                       dim_feedforward=3072,
                                                       dropout=0.1,
                                                       activation="gelu",
                                                       batch_first=True,
                                                       norm_first=True)

torch_transformer_encoder_block

In [ ]:
summary(model=torch_transformer_encoder_block,
        input_size=(1, 197, 768),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Although PyTorch has in-built transformer encoder layer, we have hard coded it for the purpose of practice and enhancing technical skills. Benefits to use pre-built PyTorch layer:
  * Less prone to errors.
  * Potential benefit to speed ups (performance boosts.)

## 8. Putting it all together to create ViT

In [ ]:
# Create a ViT class
class ViT(nn.Module):
  def __init__(self,
               img_size:int=224,
               input_channels:int=3,
               patch_size:int=16,
               embedding_dim:int=768,
               attn_heads:int=12,
               num_transformer_layers:int=12,
               mlp_size:int=3072,
               attn_dropout:int=0,
               mlp_dropout:int=0.1,
               embedding_dropout:int=0.1, # dropout for batch and embedding positions
               num_classes:int=1000): # ImageNet has 1000 classes
    super().__init__()

    # Make assertion the image size is divisible by patch size
    assert img_size % patch_size == 0, "The image size must be divisible by patch size"

    # Calculate the number of patches
    self.number_of_patches = ((img_size*2)  // patch_size**2)

    # Create leanable class embedding (needs to go at front of sequence of patch embeddings)
    self.class_embedding = nn.Parameter(torch.randn(1, 1, embedding_dim),
                                        requires_grad=True)

    # Create learnable position embedding
    self.pos_embedding = nn.Parameter(torch.randn(1,number_of_patches+1, embedding_dim),
                                      requires_grad=True)

    # Create embedding dropout value
    self.embedding_dropout = nn.Dropout(p=embedding_dropout)

    # Create patch embedding layer
    self.patch_embedding = PatchEmbedding(input_channels=3,
                                          embedding_dim=768,
                                          patch_size=16)

    # Create transformer encoder block
    self.transformer_encoder = nn.Sequential(*(TransformerEncoderBlock(embedding_dim=embedding_dim,
                                                                       num_heads=attn_heads,
                                                                       mlp_size=mlp_size,
                                                                       mlp_dropout=mlp_dropout,
                                                                       attn_dropout=attn_dropout) for _ in range(num_transformer_layers)))

    # Create classifier head
    self.classifier = nn.Sequential(
        nn.LayerNorm(normalized_shape=embedding_dim),
        nn.Linear(in_features=embedding_dim,
                  out_features=num_classes)
    )

  def forward(self, x:torch.Tensor) -> torch.Tensor:
    # Get the batch size
    batch_size = x.shape[0]

    # Create class token embedding and expand it to match the batch size
    class_token = self.class_embedding.expand(batch_size, -1, -1)

    # Create the patch embedding
    x = self.patch_embedding(x)

    # concatenate the class token
    x = torch.cat((class_token, x), dim=1)

    # Add the position embedding to the patch and classs embedding
    x = x + self.pos_embedding

    # Apply dropout on patch embedding
    x = self.embedding_dropout(x)

    # Pass position and patch embedding to transformer encoding
    x = self.transformer_encoder(x)

    # Put 0th index logit through classifier
    x = self.classifier(x[:,0])

    return x

In [ ]:
batch_size =32
embedding_dim=768
class_embedding = nn.Parameter(data=torch.randn(1,1,embedding_dim), requires_grad=True)

class_embedding_expanded = class_embedding.expand(batch_size,-1,-1)
print(class_embedding.shape)
print(class_embedding_expanded.shape)

In [ ]:
plt.imshow(rand_image_tensor.squeeze(0).permute(2,1,0))

In [ ]:
set_seeds()

# Create a random image tensor with same shape as a single image
random_image_tensor = torch.randn(1, 3, 224, 224)

# Create an instace of ViT with the number of classes we are working with (pizza, steak and sushi = 3)
vit = ViT(num_classes=len(class_names)).to(device)

# Pass the random image tensor to our ViT instance
vit(random_image_tensor.to(device))

### 8.1 Getting a visual summary of our ViT model


In [ ]:
from torchinfo import summary

summary(model= ViT(num_classes=1000),
        input_size=(1, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

In [ ]:
from torchinfo import summary

summary(model= ViT(num_classes=len(class_names)),
        input_size=(1, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

## 9. Setting up training code for our custom dataset
After replicating the function, we will train it on our custom data.

## 9.1 Creating an optimizer
The paper uses Adam optimizer with b1 value of 0.9 and B2 of 0.999 (defaults) and a weight decay of 0.1.
weight decay is a regularization technique that prevents overfitting ba adding a smaall penality, usually the L2 norm of the weights to the loss function

In [ ]:
optimizer = torch.optim.Adam(params=vit.parameters(),
                             lr=1e-3,
                             betas=(0.9, 0.999),
                             weight_decay=0.1
                             )

### 9.2 Creating a loss function
`torch.nn.CrossEntropyLoss()`

In [ ]:
loss_fn = nn.CrossEntropyLoss()

### 9.3 Training our vit model

In [ ]:
from going_modular import engine

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=vit.parameters(),
                             lr=1e-3,
                             betas=(0.9, 0.999),
                             weight_decay=0.1
                             )

results = engine.train(model=vit,
                       train_dataloader=train_dataloader,
                       test_dataloader=test_dataloader,
                       loss_fn=loss_fn,
                       optimizer=optimizer,
                       epochs=10,
                       device=device)

In [ ]:
results

### 9.4 What our training setup is missing?
Although the correct architecture replication, some reasons leaded to the poor results:
1. Prevent underfitting:
  * Data: our setup uses far less data.
2. Prevent Overvitting:
  * LR warmup - start with a low  lr and increase to a base lr.
  * lr decay - as model gets closer to convergence, start to lower the lr.
  * Gradient clipping - prevent gradients from getting too big.

### 9.5 plotting los curves for our model

In [ ]:
from helper_functions import plot_loss_curves

plot_loss_curves(results)

> we can see the model underfitting and overfitting in the same time

## 10. Using a pretrained ViT from `torchvision.models`
Generally, if you can use a pretrained model from a large datasets on our problem,
if you can find a pretrained model and use transfer learning, give it a go, it often achieve great results with little data.results

### 10.1 **Why use a pretrained mode?**
  * Sometimes data is limited.
  * Limited training results.
  * Get better results faster (sometimes).

In [ ]:
print(f"The cost of training using tpu for 30 days (30 as shared in the paper):{8*24*30}$")

### 10.2 Prepare a pretrained vit to use with our data

In [ ]:
import torch
import torchvision

# Get pretrained weights for ViT-base
pretrained_vit_weights = torchvision.models.ViT_B_16_Weights.DEFAULT

# Setup a ViT model instance with pretrined weights
pretrained_vit = torchvision.models.vit_b_16(weights=pretrained_vit_weights)
#print(pretrained_vit)

# Freeze the base parameters
for param in pretrained_vit.parameters():
  param.requires_grad=False

# Updata the classifier head
set_seeds()
pretrained_vit.heads = nn.Linear(in_features=768, out_features= len(class_names))

In [ ]:
summary(model= pretrained_vit,
        input_size=(1, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

### 10.3 Preparing data for pretrained vit
when using a pretrined model we should make sure our data is formatted in the same wat the model was trained on

In [ ]:
# Get automatic transform from pretrined vit weights
vit_transform = pretrained_vit_weights.transforms()
vit_transform

In [ ]:
# Setup dataloaders
from going_modular import data_setup
train_dataloader_pretrained, test_dataloader_pretrined, class_names = data_setup.create_dataloaders(train_dir= train_dir,
                                                                                          test_dir= test_dir,
                                                                                          transform=vit_transform,
                                                                                          batch_size=32)

In [ ]:
from going_modular import engine

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=pretrained_vit.parameters(),
                             lr=1e-3,
                             )

pretrained_vit_results = engine.train(model=pretrained_vit,
                       train_dataloader=train_dataloader_pretrained,
                       test_dataloader=test_dataloader_pretrined,
                       loss_fn=loss_fn,
                       optimizer=optimizer,
                       epochs=10,
                       device=device)

In [ ]:
plot_loss_curves(pretrained_vit_results)

### 10.6 Saving best performing model
Now we save best performing model to fil, and check filesize. We check filesize because when we deploy a model to website/app, we may have some limitations of the size.

In [ ]:
# Save the model
from going_modular import utils

utils.save_model(model=pretrained_vit,
                 target_dir="models",
                 model_name="08_pretrained_vit_feature_extractor.pth")

In [ ]:
from pathlib import Path

# Get the model size in bytes then convert to megabytes
pretraied_vit_model_size = Path("models/08_pretrained_vit_feature_extractor.pth").stat().st_size // (1024*1024)
print(f"[INFO] Pretrained ViT feature extactor model size: {pretraied_vit_model_size} MB.")

> pretrained gets the best results, but the model size is 11x larger. Large model size can cause issues when we go to deploy it as it may not the predictions as fast as smaller model.

## 11. Predicting a custom image

In [ ]:
# Download the image
import requests

data_path = Path("data")

# Setup custom image path
custom_image_path = data_path /"pizza.jpg"

# Download the image if it does not exist
if not custom_image_path.is_file():
  with open(custom_image_path, "wb") as f:
    # Download the image from GitHub
    request = requests.get("https://raw.githubusercontent.com/esraalmaeeni/pytorch-zero-to-mastery-MyPractice/main/images/pizza.jpg")
    print(f"Download {custom_image_path}...")
    f.write(request.content)
else:
  print(f"{custom_image_path} already exists, skipping downloading...")

In [ ]:
from going_modular import predictions

predictions.pred_and_plot_image(model = pretrained_vit,
                                image_path= custom_image_path,
                                class_names= class_names,
                                image_size=(224,224))

In [ ]:
#DONE!